In [1]:
from lexicoht10 import *
from sintacticoht10 import *

In [2]:
# Ejemplo de uso
codigo_fuente = """
inicio
    a = 10
    b = 20
    c = a + b * 2
    si (c > 30) entonces
        escribir(c)
        d = c - 10
    finsi
    escribir(d)
fin
"""

In [3]:
tokens = identificar_tokens(codigo_fuente)
tokens

[('KEYWORD', 'inicio'),
 ('IDENTIFIER', 'a'),
 ('OPERATOR', '='),
 ('NUMBER', '10'),
 ('IDENTIFIER', 'b'),
 ('OPERATOR', '='),
 ('NUMBER', '20'),
 ('IDENTIFIER', 'c'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'a'),
 ('OPERATOR', '+'),
 ('IDENTIFIER', 'b'),
 ('OPERATOR', '*'),
 ('NUMBER', '2'),
 ('KEYWORD', 'si'),
 ('DELIMITER', '('),
 ('IDENTIFIER', 'c'),
 ('OPERATOR', '>'),
 ('NUMBER', '30'),
 ('DELIMITER', ')'),
 ('KEYWORD', 'entonces'),
 ('KEYWORD', 'escribir'),
 ('DELIMITER', '('),
 ('IDENTIFIER', 'c'),
 ('DELIMITER', ')'),
 ('IDENTIFIER', 'd'),
 ('OPERATOR', '='),
 ('IDENTIFIER', 'c'),
 ('OPERATOR', '-'),
 ('NUMBER', '10'),
 ('KEYWORD', 'finsi'),
 ('KEYWORD', 'escribir'),
 ('DELIMITER', '('),
 ('IDENTIFIER', 'd'),
 ('DELIMITER', ')'),
 ('KEYWORD', 'fin')]

In [4]:
print('Tokens encontrados')
for tipo, valor in tokens:
    print(f'{tipo}: {valor}')

Tokens encontrados
KEYWORD: inicio
IDENTIFIER: a
OPERATOR: =
NUMBER: 10
IDENTIFIER: b
OPERATOR: =
NUMBER: 20
IDENTIFIER: c
OPERATOR: =
IDENTIFIER: a
OPERATOR: +
IDENTIFIER: b
OPERATOR: *
NUMBER: 2
KEYWORD: si
DELIMITER: (
IDENTIFIER: c
OPERATOR: >
NUMBER: 30
DELIMITER: )
KEYWORD: entonces
KEYWORD: escribir
DELIMITER: (
IDENTIFIER: c
DELIMITER: )
IDENTIFIER: d
OPERATOR: =
IDENTIFIER: c
OPERATOR: -
NUMBER: 10
KEYWORD: finsi
KEYWORD: escribir
DELIMITER: (
IDENTIFIER: d
DELIMITER: )
KEYWORD: fin


In [5]:
try:
    print('\nIniciando analisis sintactico...')
    parser = Parser(tokens)
    arbol_ast = parser.parsear()
    print('Analisis sintactico completo sin errores!')
except SyntaxError as e:
    print(e)


Iniciando analisis sintactico...
Analisis sintactico completo sin errores!


In [6]:
import json 

In [7]:
def imprimir_ast(nodo):
    if isinstance(nodo, NodoPrograma):
        return {
            'tipo': 'Programa',
            'funciones': [imprimir_ast(f) for f in nodo.funciones],
            'main': imprimir_ast(nodo.main)
        }
    elif isinstance(nodo, NodoFuncion):
        return {
            'tipo': 'Funcion',
            'nombre': nodo.nombre[1],
            'parametros': [imprimir_ast(p) for p in nodo.parametros],
            'cuerpo': [imprimir_ast(c) for c in nodo.cuerpo]
        }
    elif isinstance(nodo, NodoParametro):
        return {'tipo': 'Parametro', 'nombre': nodo.nombre[1]}
    elif isinstance(nodo, NodoAsignacion):
        return {
            'tipo': 'Asignacion',
            'variable': nodo.nombre[1],
            'expresion': imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoOperacion):
        return {
            'tipo': 'Operacion',
            'operador': nodo.operador[1],
            'izquierda': imprimir_ast(nodo.izquierda),
            'derecha': imprimir_ast(nodo.derecha)
        }
    elif isinstance(nodo, NodoCondicional):
        return {
            'tipo': 'Condicional',
            'condicion': imprimir_ast(nodo.condicion),
            'cuerpo': [imprimir_ast(c) for c in nodo.cuerpo]
        }
    elif isinstance(nodo, NodoEscribir):
        return {
            'tipo': 'Escribir',
            'argumentos': [imprimir_ast(a) for a in nodo.argumentos]
        }
    elif isinstance(nodo, NodoRetorno):
        return {'tipo': 'Retorno', 'valor': imprimir_ast(nodo.expresion)}
    elif isinstance(nodo, NodoIdentificador):
        return {'tipo': 'Identificador', 'nombre': nodo.nombre[1]}
    elif isinstance(nodo, NodoNumero):
        return {'tipo': 'Numero', 'valor': nodo.valor[1]}
    elif isinstance(nodo, NodoCadena):
        return {'tipo': 'Cadena', 'valor': nodo.valor[1]}
    elif isinstance(nodo, NodoLlamadaFuncion):
        return {
            'tipo': 'LlamadaFuncion',
            'funcion': nodo.nombre_funcion,
            'argumentos': [imprimir_ast(a) for a in nodo.argumentos]
        }
    return {}

In [8]:
print(codigo_fuente)


inicio
    a = 10
    b = 20
    c = a + b * 2
    si (c > 30) entonces
        escribir(c)
        d = c - 10
    finsi
    escribir(d)
fin



In [9]:
codigo = arbol_ast.generarCodigo()

In [10]:
print(codigo)

section .bss
    a: resd 1
    b: resd 1
    c: resd 1
section .text
global _start
_start:
main:

    mov eax, 10
    mov [a], eax

    mov eax, 20
    mov [b], eax

    mov eax, [a]
    push eax

    mov eax, [b]
    mov  ebx, eax
    pop  eax
    add eax, ebx
    push eax

    mov eax, 2
    mov  ebx, eax
    pop  eax
    imul ebx
    mov [c], eax

    mov eax, [c]
    push eax

    mov eax, 30
    mov  ebx, eax
    pop  eax
    cmp  eax, ebx
    setg al
    movzx eax, al
    cmp eax, 0
    je  .fin_si

    mov eax, [c]
    push eax
    mov  eax, 4
    mov  ebx, 1
    int  0x80
    pop  eax

    mov eax, [c]
    push eax

    mov eax, 10
    mov  ebx, eax
    pop  eax
    sub eax, ebx
    mov [d], eax
.fin_si:

    mov eax, [d]
    push eax
    mov  eax, 4
    mov  ebx, 1
    int  0x80
    pop  eax
    ret

    mov eax, 1
    xor ebx, ebx
    int 0x80


In [11]:
import sematicoht10

In [12]:
analizador_semantico = sematicoht10.AnalizadorSemantico()
analizador_semantico.analizar(arbol_ast)

Advertencia: 'd' puede no estar inicializada


In [13]:
analizador_semantico.tabla_simbolos.funciones

{}

In [14]:
analizador_semantico.tabla_simbolos.variables 

{'a': 'int', 'b': 'int', 'c': 'int', 'd': 'int'}

In [15]:
nodo_exp = NodoOperacion(NodoNumero(('NUMBER','1')),('OPERATOR','*'), NodoIdentificador(('IDENTIFIER','algo')))
print(json.dumps(imprimir_ast(nodo_exp), indent=1))

{
 "tipo": "Operacion",
 "operador": "*",
 "izquierda": {
  "tipo": "Numero",
  "valor": "1"
 },
 "derecha": {
  "tipo": "Identificador",
  "nombre": "algo"
 }
}


In [16]:
exp_op = nodo_exp.optimizar()
print(json.dumps(imprimir_ast(exp_op), indent=1))

{
 "tipo": "Identificador",
 "nombre": "algo"
}


In [17]:
import subprocess
def compilar(programa):
    # Generar el codigo en ensamblador
    codigo_asm = programa.generarCodigo()
    print(codigo_asm)
    text_file = open("ejasmcompiladores.asm", "w")
    text_file.write(codigo_asm)
    text_file.close()
    subprocess.run(["nasm", "-f", "elf", "ejasmcompiladores.asm"])
    subprocess.run(["ld", "-m", "elf_i386", "ejasmcompiladores.o", "-o", "ejasmcompiladores"])

In [18]:
compilar(arbol_ast)

section .bss
    a: resd 1
    b: resd 1
    c: resd 1
section .text
global _start
_start:
main:

    mov eax, 10
    mov [a], eax

    mov eax, 20
    mov [b], eax

    mov eax, [a]
    push eax

    mov eax, [b]
    mov  ebx, eax
    pop  eax
    add eax, ebx
    push eax

    mov eax, 2
    mov  ebx, eax
    pop  eax
    imul ebx
    mov [c], eax

    mov eax, [c]
    push eax

    mov eax, 30
    mov  ebx, eax
    pop  eax
    cmp  eax, ebx
    setg al
    movzx eax, al
    cmp eax, 0
    je  .fin_si

    mov eax, [c]
    push eax
    mov  eax, 4
    mov  ebx, 1
    int  0x80
    pop  eax

    mov eax, [c]
    push eax

    mov eax, 10
    mov  ebx, eax
    pop  eax
    sub eax, ebx
    mov [d], eax
.fin_si:

    mov eax, [d]
    push eax
    mov  eax, 4
    mov  ebx, 1
    int  0x80
    pop  eax
    ret

    mov eax, 1
    xor ebx, ebx
    int 0x80


ejasmcompiladores.asm:57: error: symbol `d' not defined
ejasmcompiladores.asm:58: error: label `main.fin_si' changed during code generation (offset 0x77 -> 0x72) [-w+error=label-redef-late]
ejasmcompiladores.asm:60: error: symbol `d' not defined
ld: file cannot be open()ed, errno=2 path=elf_i386
